In [1]:
# import relevant libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


In [2]:
# 1. Simulate Synthetic M-Pesa Transaction Stream
np.random.seed(42)
n_txns = 5000

data = {
    'txn_id': [f"TXN_{i:05d}" for i in range(n_txns)],
    'amount_kes': np.random.exponential(scale=2500, size=n_txns).round(2),
    'hour_of_day': np.random.uniform(0, 24, size=n_txns).round(2),
    'txn_velocity_1hr': np.random.poisson(lam=1.2, size=n_txns),
    'is_new_agent': np.random.choice([0, 1], size=n_txns, p=[0.95, 0.05])
}
df = pd.DataFrame(data)

In [3]:
print(df)


         txn_id  amount_kes  hour_of_day  txn_velocity_1hr  is_new_agent
0     TXN_00000     1173.17         9.45                 1             0
1     TXN_00001     7525.30        11.36                 0             0
2     TXN_00002     3291.86        20.51                 1             0
3     TXN_00003     2282.36         8.16                 1             0
4     TXN_00004      424.06        20.87                 4             0
...         ...         ...          ...               ...           ...
4995  TXN_04995     5692.23        20.58                 2             0
4996  TXN_04996      317.83        21.54                 2             0
4997  TXN_04997      993.16        22.72                 1             0
4998  TXN_04998     4228.59         9.54                 0             0
4999  TXN_04999     2273.99         5.21                 0             0

[5000 rows x 5 columns]


In [4]:
# 2. Inject Fraud Scenarios (SIM-Swap Drain & Structuring Patterns)
# Scenario A: High velocity + high amount at odd hours
df.loc[100:105, ['amount_kes', 'hour_of_day', 'txn_velocity_1hr']] = [145000.00, 3.15, 12]
# Scenario B: Rapid cash out via unverified agent
df.loc[2500:2503, ['amount_kes', 'txn_velocity_1hr', 'is_new_agent']] = [49500.00, 9, 1]


In [5]:
# 3. Feature Selection & Preprocessing
features = ['amount_kes', 'hour_of_day', 'txn_velocity_1hr', 'is_new_agent']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [6]:
# 4. Fit Isolation Forest Anomaly Detector
# contamination=0.01 assumes ~1% of transactions are fraudulent/anomalous
model = IsolationForest(contamination=0.01, random_state=42)
df['anomaly_score'] = model.fit_predict(X_scaled)

In [7]:
df


,txn_id,amount_kes,hour_of_day,txn_velocity_1hr,is_new_agent,anomaly_score
0,TXN_00000,1173.17,9.45,1,0,1
1,TXN_00001,7525.30,11.36,0,0,1
2,TXN_00002,3291.86,20.51,1,0,1
3,TXN_00003,2282.36,8.16,1,0,1
4,TXN_00004,424.06,20.87,4,0,1
...,...,...,...,...,...,...
4995,TXN_04995,5692.23,20.58,2,0,1
4996,TXN_04996,317.83,21.54,2,0,1
4997,TXN_04997,993.16,22.72,1,0,1
4998,TXN_04998,4228.59,9.54,0,0,1


In [8]:
# Map output: -1 indicates an anomaly (fraud flag), 1 indicates normal transaction
df['fraud_flag'] = np.where(df['anomaly_score'] == -1, "High Risk", "Normal")

# 5. Review Flagged Exploits
high_risk_txns = df[df['fraud_flag'] == "High Risk"]
print(f"🚨 Total High-Risk Transactions Flagged: {len(high_risk_txns)}")
print(high_risk_txns[['txn_id', 'amount_kes', 'hour_of_day', 'txn_velocity_1hr', 'fraud_flag']].head(10))

🚨 Total High-Risk Transactions Flagged: 50
        txn_id  amount_kes  hour_of_day  txn_velocity_1hr fraud_flag
100  TXN_00100   145000.00         3.15                12  High Risk
101  TXN_00101   145000.00         3.15                12  High Risk
102  TXN_00102   145000.00         3.15                12  High Risk
103  TXN_00103   145000.00         3.15                12  High Risk
104  TXN_00104   145000.00         3.15                12  High Risk
105  TXN_00105   145000.00         3.15                12  High Risk
210  TXN_00210     1989.24        22.74                 4  High Risk
518  TXN_00518    10910.69         2.74                 4  High Risk
531  TXN_00531    20431.11         3.20                 5  High Risk
843  TXN_00843     8115.00         0.40                 0  High Risk
